In [1]:
import sqlite3

conn = sqlite3.connect("permits.db")

In [3]:
import pandas as pd

# =========================
# LOAD FILES
# =========================
permits = pd.read_csv("permits.csv", engine="python", encoding="latin1", on_bad_lines="skip")
approved = pd.read_csv("approved_permits.csv", engine="python", encoding="latin1", on_bad_lines="skip")
requests_311 = pd.read_csv("requests_311.csv", engine="python", encoding="latin1", on_bad_lines="skip")

print("Files loaded")

Files loaded


In [4]:
# =========================
# CLEAN + CREATE METRICS
# =========================
permits['Filing Date'] = pd.to_datetime(permits['Filing Date'], errors='coerce')
permits['Issuance Date'] = pd.to_datetime(permits['Issuance Date'], errors='coerce')

approved['Approved Date'] = pd.to_datetime(approved['Approved Date'], errors='coerce')
approved['Issued Date'] = pd.to_datetime(approved['Issued Date'], errors='coerce')

requests_311['Created Date'] = pd.to_datetime(requests_311['Created Date'], errors='coerce')
requests_311['Closed Date'] = pd.to_datetime(requests_311['Closed Date'], errors='coerce')

permits['days_filing_to_issuance'] = (permits['Issuance Date'] - permits['Filing Date']).dt.days
approved['days_approved_to_issued'] = (approved['Issued Date'] - approved['Approved Date']).dt.days
requests_311['days_to_close'] = (requests_311['Closed Date'] - requests_311['Created Date']).dt.days

print("Metrics created")

/tmp/ipykernel_28235/1344002647.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  approved['Approved Date'] = pd.to_datetime(approved['Approved Date'], errors='coerce')
/tmp/ipykernel_28235/1344002647.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  approved['Issued Date'] = pd.to_datetime(approved['Issued Date'], errors='coerce')
/tmp/ipykernel_28235/1344002647.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  requests_311['Created Date'] = pd.to_datetime(requests_311['Created Date'], errors='coerce')


Metrics created


/tmp/ipykernel_28235/1344002647.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  requests_311['Closed Date'] = pd.to_datetime(requests_311['Closed Date'], errors='coerce')


In [6]:
# =========================
# BOROUGH SUMMARY
# =========================
permits['BOROUGH'] = permits['BOROUGH'].str.title()
approved['Borough'] = approved['Borough'].str.title()
requests_311['Borough'] = requests_311['Borough'].str.title()

permits_summary = permits.groupby('BOROUGH').agg(
    total_permits=('Job #', 'count'),
    avg_days_filing_to_issuance=('days_filing_to_issuance', 'mean')
).reset_index()

approved_summary = approved.groupby('Borough').agg(
    total_approved=('Job Filing Number', 'count'),
    avg_days_approved_to_issued=('days_approved_to_issued', 'mean')
).reset_index()

requests_summary = requests_311.groupby('Borough').agg(
    total_complaints=('Unique Key', 'count'),
    avg_days_to_close=('days_to_close', 'mean')
).reset_index()

final_summary = permits_summary.merge(
    approved_summary,
    left_on='BOROUGH',
    right_on='Borough',
    how='outer'
)

final_summary = final_summary.merge(
    requests_summary,
    left_on='BOROUGH',
    right_on='Borough',
    how='outer'
)

final_summary = final_summary.drop(columns=['Borough_x', 'Borough_y'], errors='ignore')

print("Final summary ready")

Final summary ready


In [7]:
import sqlite3
import pandas as pd

# =========================
# 1. CREATE SQLITE DATABASE
# =========================
conn = sqlite3.connect("nyc_permit_analysis.db")

# =========================
# 2. SAVE DATAFRAMES TO SQL
# =========================
permits.to_sql("permits", conn, if_exists="replace", index=False)
approved.to_sql("approved_permits", conn, if_exists="replace", index=False)
requests_311.to_sql("requests_311", conn, if_exists="replace", index=False)
final_summary.to_sql("final_summary", conn, if_exists="replace", index=False)

# =========================
# 3. CHECK TABLES
# =========================
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn
)

print("TABLES CREATED:")
display(tables)

# =========================
# 4. SAMPLE SQL QUERIES
# =========================
query1 = """
SELECT
    BOROUGH,
    total_permits,
    ROUND(avg_days_filing_to_issuance, 2) AS avg_days_filing_to_issuance
FROM final_summary
ORDER BY avg_days_filing_to_issuance DESC;
"""
sql_permits = pd.read_sql_query(query1, conn)

query2 = """
SELECT
    BOROUGH,
    total_approved,
    ROUND(avg_days_approved_to_issued, 2) AS avg_days_approved_to_issued
FROM final_summary
ORDER BY avg_days_approved_to_issued DESC;
"""
sql_approved = pd.read_sql_query(query2, conn)

query3 = """
SELECT
    BOROUGH,
    total_complaints,
    ROUND(avg_days_to_close, 2) AS avg_days_to_close
FROM final_summary
ORDER BY avg_days_to_close DESC;
"""
sql_complaints = pd.read_sql_query(query3, conn)

query4 = """
SELECT
    BOROUGH,
    ROUND(avg_days_approved_to_issued, 2) AS avg_days_approved_to_issued,
    total_complaints,
    ROUND(avg_days_to_close, 2) AS avg_days_to_close
FROM final_summary
ORDER BY total_complaints DESC;
"""
sql_delay_vs_complaints = pd.read_sql_query(query4, conn)

print("\nSQL QUERY 1: PERMIT TIMELINE BY BOROUGH")
display(sql_permits)

print("\nSQL QUERY 2: APPROVAL DELAYS BY BOROUGH")
display(sql_approved)

print("\nSQL QUERY 3: 311 COMPLAINTS BY BOROUGH")
display(sql_complaints)

print("\nSQL QUERY 4: DELAYS VS COMPLAINTS")
display(sql_delay_vs_complaints)

# =========================
# 5. EXPORT SQL RESULTS
# =========================
sql_permits.to_csv("sql_permits_summary.csv", index=False)
sql_approved.to_csv("sql_approved_summary.csv", index=False)
sql_complaints.to_csv("sql_complaints_summary.csv", index=False)
sql_delay_vs_complaints.to_csv("sql_delay_vs_complaints.csv", index=False)

print("SQL query outputs exported.")

TABLES CREATED:


,name
0,approved_permits
1,final_summary
2,permits
3,requests_311



SQL QUERY 1: PERMIT TIMELINE BY BOROUGH


,BOROUGH,total_permits,avg_days_filing_to_issuance
0,Bronx,3659.0,11.21
1,Manhattan,5278.0,9.06
2,Brooklyn,8142.0,7.92
3,Queens,7781.0,6.52
4,Staten Island,2464.0,3.79
5,None,NaN,NaN



SQL QUERY 2: APPROVAL DELAYS BY BOROUGH


...


SQL QUERY 3: 311 COMPLAINTS BY BOROUGH


,BOROUGH,total_complaints,avg_days_to_close
0,Queens,7797,112.37
1,Staten Island,992,69.06
2,Brooklyn,7691,64.66
3,Bronx,4439,52.21
4,Manhattan,4826,29.29
5,None,9,17.33



SQL QUERY 4: DELAYS VS COMPLAINTS


,BOROUGH,avg_days_approved_to_issued,total_complaints,avg_days_to_close
0,Queens,233.89,7797,112.37
1,Brooklyn,252.26,7691,64.66
2,Manhattan,189.03,4826,29.29
3,Bronx,268.11,4439,52.21
4,Staten Island,266.07,992,69.06
5,None,NaN,9,17.33


SQL query outputs exported.


In [8]:
final_summary.to_csv("final_summary.csv", index=False)
print("final_summary.csv exported")

final_summary.csv exported
